# 01 — SD1.5 full pipeline

This is the main statistical baseline. Use `SMOKE_TEST=True` for a fast GPU check, then set it to `False` for thesis-scale runs.

In [ ]:
from pathlib import Path
import os, sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.environ["PYTHONPATH"] = str(PROJECT_ROOT / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
print("PROJECT_ROOT =", PROJECT_ROOT)

from diffusion_attention_analysis_v2.notebook_runner import run_config, run_suite, show_report

In [ ]:
SMOKE_TEST = True
CAPTURE_OVERRIDES = ["data.max_prompts=8", "runtime.num_seeds=1"] if SMOKE_TEST else []
SAE_OVERRIDES = ["sae.num_epochs=1", "sae.max_activation_files=32", "sae.max_tokens_per_file=1024"] if SMOKE_TEST else []
INTERVENTION_OVERRIDES = ["data.max_prompts=2", "runtime.num_seeds=1"] if SMOKE_TEST else []

## Stage 1 — capture

In [ ]:
run_config("configs/01_capture/sd15_full.yaml", overrides=CAPTURE_OVERRIDES)

## Stage 2 — attention localization

This stage requires masks in `annotations/`. If masks are absent, it will skip and write a report.

In [ ]:
run_config("configs/02_attention_localization/sd15_full.yaml")
show_report("outputs/sd15/02_attention_localization/report.json")

## Stage 3 — SAE training

In [ ]:
run_config("configs/03_sae_training/sd15_full.yaml", overrides=SAE_OVERRIDES)
show_report("outputs/sd15/03_sae_mid/report.json")

## Stage 4 — concept dictionary

Requires masks and a trained SAE.

In [ ]:
run_config("configs/04_concept_dictionary/sd15_full.yaml", overrides=(["dictionary.max_activation_files=32"] if SMOKE_TEST else []))
show_report("outputs/sd15/04_concept_dictionary/report.json")

## Stage 5a — map-level intervention

In [ ]:
run_config("configs/05_interventions/sd15_map_reweight.yaml", overrides=INTERVENTION_OVERRIDES)
show_report("outputs/sd15/05_intervention_map_reweight/report.json")

## Stage 5b — SAE steering

In [ ]:
run_config("configs/05_interventions/sd15_sae_steering.yaml", overrides=INTERVENTION_OVERRIDES)
show_report("outputs/sd15/05_intervention_sae_steering/report.json")